# CIFAR-100 CLS-CI Pilot Notebook

Runs the dual-substrate CLS architecture (hippocampe + neocortex, multi-level anchor + interleaved replay + separate consolidation phase) on Split-CIFAR-100 class-incremental.

## Prerequisites

- **Colab Pro** with an L4 or T4 GPU (L4 is faster; ~2-3h for the full T=10 n=3 pilot).
- `Runtime → Change runtime type → GPU` (L4 if available).

## What's in this notebook

1. Clone the repo + install pinned deps.
2. Verify the runtime sees a GPU.
3. Pre-cache the CIFAR-100 HuggingFace dataset (one-time, ~10s).
4. Quick smoke (T=2, 2 epochs, ~1-2 min on GPU) — verifies the pipeline.
5. (Commented) Full pilot launcher — T=10, 30 epochs/task, n=3 seeds, ~2-3h on L4.
6. Results plotting placeholder.

## Resuming after a disconnect

The training script writes a checkpoint at the end of every task into `results/checkpoints/cifar100_ci/`. Pass `--resume` on the next run with the same `--config_name` and `--n_seeds` to pick up where you left off.


## 1. Clone the repo

In [ ]:
!git clone https://github.com/frpatry/continual-synapse-layer.git
%cd continual-synapse-layer

## 2. Install pinned dependencies

Colab usually has a recent torch already; we install only what's missing.  If your runtime's torch differs from the pinned `torch==2.12.0`, the project still works (PyTorch API stable across that range) but a perfectly-reproducible run uses the pin.

In [ ]:
!pip install -q datasets==4.0.0 scikit-learn==1.8.0
# torch / numpy / scipy / matplotlib / seaborn / tqdm / pillow are
# almost always preinstalled on Colab; uncomment if any import
# fails below:
# !pip install -q torch==2.12.0 numpy==2.4.6 scipy==1.16.3 matplotlib==3.10.7 seaborn==0.13.2 tqdm==4.67.1 pillow==12.2.0

## 3. GPU verification

In [ ]:
import torch

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    total_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"Memory: {total_mem_gb:.1f} GB")
else:
    print("\n⚠️  No GPU detected — switch the runtime to GPU before continuing:")
    print("     Runtime → Change runtime type → GPU (L4 if available)")

## 4. Pre-cache CIFAR-100

Triggers the one-time HuggingFace download (~170 MB).  Subsequent loads inside the training script hit this cache and complete in ~10 s.

In [ ]:
from datasets import load_dataset

ds = load_dataset("uoft-cs/cifar100")
print(f"Train: {len(ds['train']):,} samples,  Test: {len(ds['test']):,} samples.")

## 5. Smoke test (train first 2 tasks of the standard 10-task split, ~1-2 min on L4)

Same mechanical-check smoke validated locally on CPU.  Verifies:

- Models instantiate on GPU and forward/backward without NaN.
- Memory grows correctly across tasks.
- Both per-task training (with interleaved replay) and the separate consolidation phase run cleanly.
- End-of-task checkpoint is written.

`--num_tasks 2` here means **train the first 2 tasks of the standard 10-tasks-of-10-classes layout** (so the model sees classes 0-19), not 'split CIFAR-100 into 2 mega-tasks of 50 classes'. The benchmark structure is always T=10 × 10 classes via `--bench_num_tasks 10`; `--num_tasks` caps the training loop only. Expected: NEO ACC > 0.10 on the 20 task-0+task-1 classes (chance = 0.05; threshold = 2× chance = 0.10).

In [ ]:
!python experiments/45_cls_ci_cifar.py \
    --smoke \
    --num_tasks 2 \
    --bench_num_tasks 10 \
    --epochs_per_task 2 \
    --batch_size 128 \
    --n_seeds 1 \
    --num_workers 2 \
    --config_name cls_ci_v2_cifar_smoke

## 6. Full T=10 pilot (UNCOMMENT WHEN READY)

ETA on L4: ~2-3 h for n=3 seeds.  Writes a checkpoint at the end of every task per seed under `results/checkpoints/cifar100_ci/`, so a Colab disconnect costs at most one task's worth of compute (rerun with `--resume` to continue).

Uncomment the cell below when the smoke above passes and you're ready to commit compute.

In [ ]:
# !python experiments/45_cls_ci_cifar.py \
#     --num_tasks 10 \
#     --epochs_per_task 30 \
#     --batch_size 128 \
#     --n_seeds 3 \
#     --num_workers 2 \
#     --lambda_replay_inline 1.0 \
#     --cons_epochs 2 \
#     --config_name cls_ci_v2_cifar_pilot \
#     --resume

## 6b. DER baseline — apples-to-apples comparator (~30 min on L4)

Pure DER (Buzzega et al., 2020): same Reduced ResNet-18 as the CLS neocortex, same optimizer, same lr=0.05, same grad clip, same buffer size 5000, same replay batch 32, same 30 epochs/task — only the *architectural* mechanism differs (single model with MSE-on-logits replay vs the dual-system / multi-level anchor / consolidation phase).

Run this **after** the CLS pilot above completes (or in parallel if you have two GPUs, but not on the same one).

**The critical diagnostic** is whether DER also lands at ~0.17 final ACC on these exact hyperparameters:

- DER ~0.17 too → bottleneck is the training config (undertrained ResNet-18 at 30 epochs/task / lr=0.05); both architectures are limited by it. Tune training, not architecture.
- DER lands at 0.30+ → the CLS architecture has a specific issue on this benchmark that DER doesn't. Architecture-side debug needed.


In [ ]:
# Pure DER baseline — same hyperparameters as the CLS pilot above.
# Uncomment when CLS pilot has finished.
#
# !python experiments/46_der_baseline_cifar100_ci.py \
#     --num_tasks 10 \
#     --epochs_per_task 30 \
#     --batch_size 128 \
#     --n_seeds 3 \
#     --num_workers 2 \
#     --lambda_replay 1.0 \
#     --config_name der_baseline_cifar \
#     --output-dir /content/drive/MyDrive/cls_logs \
#     --checkpoint_dir /content/drive/MyDrive/cls_ckpts \
#     --resume

## 6c. Longer-training probe — 100 epochs/task at n=1 (directional signal)

Goal: if both 30-epoch pilots land at low ACC (e.g. ~0.17 for CLS, similar for DER), the most likely cause is that ResNet-18 on CIFAR-100 needs more than 30 epochs/task to actually fit each task. This probe tests that hypothesis cheaply with n=1.

Distinct `--config_name` per run so the 100ep checkpoints don't overwrite the 30ep ones in Drive.

**ETA on L4**: ~50 min for CLS, ~30 min for DER (n=1). Total ~80 min for both.

### Decision matrix

| CLS @ 100ep | DER @ 100ep | Interpretation |
|---|---|---|
| ≥ 0.40 | ≥ 0.40 | Undertraining was *the* bottleneck — scale to n=3 with this config |
| 0.25-0.35 | 0.35-0.45 | Undertraining helps but CLS has a specific weakness; debug masked KL |
| < 0.25 | < 0.30 | Other config issue (lr schedule, augmentation); deeper tuning sweep needed |
| Both jump similarly | — | Architecture comparison holds under proper training |


In [ ]:
# CLS-CI v2 @ 100 epochs/task, n=1 (directional signal)
# Uncomment to run.
#
# !python experiments/45_cls_ci_cifar.py \
#     --num_tasks 10 \
#     --epochs_per_task 100 \
#     --batch_size 128 \
#     --n_seeds 1 \
#     --num_workers 2 \
#     --lambda_replay_inline 1.0 \
#     --cons_epochs 2 \
#     --config_name cls_ci_v2_cifar_100ep \
#     --output-dir /content/drive/MyDrive/cls_logs \
#     --checkpoint_dir /content/drive/MyDrive/cls_ckpts \
#     --resume

In [ ]:
# DER baseline @ 100 epochs/task, n=1 (directional signal)
# Uncomment to run AFTER the CLS 100ep probe above finishes.
#
# !python experiments/46_der_baseline_cifar100_ci.py \
#     --num_tasks 10 \
#     --epochs_per_task 100 \
#     --batch_size 128 \
#     --n_seeds 1 \
#     --num_workers 2 \
#     --lambda_replay 1.0 \
#     --config_name der_baseline_cifar_100ep \
#     --output-dir /content/drive/MyDrive/cls_logs \
#     --checkpoint_dir /content/drive/MyDrive/cls_ckpts \
#     --resume

## 6d. Tuning sweep — 4 configs at n=1 (~1.5-2h on L4)

The 100-epoch probe revealed that simply training longer doesn't help (task 1 ACC ≈ 0.283 vs 30ep's 0.263). Hypothesis: more per-task training intensity requires stronger consolidation mechanisms to counterbalance.

Four configs sweep two axes (`lambda_replay_inline`, `cons_epochs`) on top of two epoch budgets:

| Config | epochs | λ_replay_inline | cons_epochs | What it tests |
|---|---|---|---|---|
| A | 50 | 1.0 | 2 | 50ep baseline — is 50 a sweet spot vs 30/100? |
| B | 50 | 2.0 | 2 | Stronger inline replay at moderate budget |
| C | 50 | 1.0 | 4 | More 'sleep' consolidation at moderate budget |
| D | 100 | 2.0 | 4 | All knobs up — does CLS scale with training intensity? |

Each run is n=1 (directional signal). Distinct `--config_name` per run keeps checkpoints separated in Drive. ETA: ~22 min × 3 (50ep) + ~45 min (100ep) ≈ **1.8 h total**.

### Decision after sweep

- **A best** → 50ep is the right budget; current config was just slightly under-trained
- **B or C best at 50ep** → CL hyperparam imbalance was the issue; stronger consolidation matters
- **D best** → CLS needs BOTH more training AND stronger consolidation in lockstep
- **All similar (within ~3pp)** → architectural ceiling under current design; need a different lever


In [ ]:
# Config A: 50ep baseline (cls_ci_v2 default lambdas)
!python experiments/45_cls_ci_cifar.py \
    --num_tasks 10 \
    --epochs_per_task 50 \
    --batch_size 128 \
    --n_seeds 1 \
    --num_workers 2 \
    --lambda_replay_inline 1.0 \
    --cons_epochs 2 \
    --config_name cls_50ep_L1_C2 \
    --output-dir /content/drive/MyDrive/cls_logs \
    --checkpoint_dir /content/drive/MyDrive/cls_ckpts \
    --resume

In [ ]:
# Config B: 50ep, stronger inline replay (λ=2.0)
!python experiments/45_cls_ci_cifar.py \
    --num_tasks 10 \
    --epochs_per_task 50 \
    --batch_size 128 \
    --n_seeds 1 \
    --num_workers 2 \
    --lambda_replay_inline 2.0 \
    --cons_epochs 2 \
    --config_name cls_50ep_L2_C2 \
    --output-dir /content/drive/MyDrive/cls_logs \
    --checkpoint_dir /content/drive/MyDrive/cls_ckpts \
    --resume

In [ ]:
# Config C: 50ep, more consolidation epochs (cons_epochs=4)
!python experiments/45_cls_ci_cifar.py \
    --num_tasks 10 \
    --epochs_per_task 50 \
    --batch_size 128 \
    --n_seeds 1 \
    --num_workers 2 \
    --lambda_replay_inline 1.0 \
    --cons_epochs 4 \
    --config_name cls_50ep_L1_C4 \
    --output-dir /content/drive/MyDrive/cls_logs \
    --checkpoint_dir /content/drive/MyDrive/cls_ckpts \
    --resume

In [ ]:
# Config D: 100ep + balanced lambdas (everything dialled up)
!python experiments/45_cls_ci_cifar.py \
    --num_tasks 10 \
    --epochs_per_task 100 \
    --batch_size 128 \
    --n_seeds 1 \
    --num_workers 2 \
    --lambda_replay_inline 2.0 \
    --cons_epochs 4 \
    --config_name cls_100ep_L2_C4 \
    --output-dir /content/drive/MyDrive/cls_logs \
    --checkpoint_dir /content/drive/MyDrive/cls_ckpts \
    --resume

### Sweep comparison plot

Finds every CLS JSON in Drive, groups by `config_name`, plots one line per config + prints final-ACC summary table. Run after some/all sweep configs have finished.


In [ ]:
import json
import statistics
from pathlib import Path

import matplotlib.pyplot as plt

log_dir = Path("/content/drive/MyDrive/cls_logs")
if not log_dir.exists():
    log_dir = Path("results/logs/split_cifar100_ci")

# Group most-recent CLS log per config_name.
by_config: dict[str, Path] = {}
for p in sorted(log_dir.glob("*45_cifar_cls_ci*.json")):
    if "smoke" in p.name:
        continue
    with p.open() as f:
        d = json.load(f)
    cfg = d.get("config", {}).get("config_name", "unknown")
    # Keep latest per config_name.
    if cfg not in by_config or p.stat().st_mtime > by_config[cfg].stat().st_mtime:
        by_config[cfg] = p

if not by_config:
    print("No CLS logs found in", log_dir)
else:
    plt.figure(figsize=(10, 5))
    rows = []
    for cfg, p in sorted(by_config.items()):
        with p.open() as f:
            d = json.load(f)
        seeds = d["per_seed"]
        n_tasks = len(seeds[0]["per_task"])
        means = [
            statistics.fmean(s["per_task"][t]["neo_eval_acc"] for s in seeds)
            for t in range(n_tasks)
        ]
        plt.plot(range(n_tasks), means, marker="o", label=f"{cfg} (n={len(seeds)})")
        final = statistics.fmean(s["final_neo_acc"] for s in seeds)
        cfg_dict = d.get("config", {})
        rows.append({
            "config": cfg,
            "epochs": cfg_dict.get("epochs_per_task"),
            "lambda_replay": cfg_dict.get("lambda_replay_inline"),
            "cons_epochs": cfg_dict.get("cons_epochs"),
            "n_seeds": len(seeds),
            "final_neo_acc": final,
        })
    plt.xlabel("Task index (after training)")
    plt.ylabel("Class-incremental ACC on classes seen so far")
    plt.title("CLS-CI v2 sweep — per-task ACC by config")
    plt.legend(loc="best", fontsize=8)
    plt.grid(alpha=0.3)
    plt.show()

    print("\nFinal NEO ACC summary:")
    hdr = f"{'config':<24} {'epochs':>6} {'λ_replay':>9} {'cons_ep':>7} {'n':>2} {'final ACC':>10}"
    print(hdr); print("-" * len(hdr))
    for r in sorted(rows, key=lambda x: -x['final_neo_acc']):
        print(
            f"{r['config']:<24} {r['epochs']!s:>6} "
            f"{r['lambda_replay']!s:>9} {r['cons_epochs']!s:>7} "
            f"{r['n_seeds']:>2} {r['final_neo_acc']:>10.3f}"
        )

## 6e. Three-pivot probe — different levers (~65 min on L4)

Config A vs B (50ep) showed `λ_replay_inline` adjustments don't break the ~0.17 plateau. Try qualitatively different levers:

| Pivot | Lever | Hypothesis |
|---|---|---|
| 1 | `--lr 0.1 --warmup_epochs 5` | Conservative lr (0.05) was leaving capacity on the table; with warmup the literature-standard 0.1 should be stable and faster-learning |
| 2 | `--no_masked_kl` | Masked KL renormalisation is the bug — full 100-class KL gives a stronger consolidation pull on the neo |
| 3 | `--neocortex_arch small_cnn` | The ResNet-18 / hipp capacity ratio is the issue; matching scales (small CNN both substrates) may work better, like the MLP recipe |

Each n=1. Distinct `--config_name`. ETA: ~22 min each = **~65 min total** on L4.

### Decision criteria

- **Any pivot ≥ 0.25 final ACC** → that direction IS the lever, scale to n=3
- **All stay at 0.16-0.18** → architectural limit for current design; accept and document
- **Pivot 2 (no masked KL) major jump** → masked KL was the bug all along


In [ ]:
# Pivot 1: LR=0.1 with 5-epoch warmup (per task), epochs=50
!python experiments/45_cls_ci_cifar.py \
    --num_tasks 10 \
    --epochs_per_task 50 \
    --batch_size 128 \
    --n_seeds 1 \
    --num_workers 2 \
    --lr 0.1 \
    --warmup_epochs 5 \
    --lambda_replay_inline 1.0 \
    --cons_epochs 2 \
    --config_name cls_50ep_lr0.1_wu5 \
    --output-dir /content/drive/MyDrive/cls_logs \
    --checkpoint_dir /content/drive/MyDrive/cls_ckpts \
    --resume

In [ ]:
# Pivot 2: standard lr=0.05 + full KL (no masking)
!python experiments/45_cls_ci_cifar.py \
    --num_tasks 10 \
    --epochs_per_task 50 \
    --batch_size 128 \
    --n_seeds 1 \
    --num_workers 2 \
    --lr 0.05 \
    --no_masked_kl \
    --lambda_replay_inline 1.0 \
    --cons_epochs 2 \
    --config_name cls_50ep_fullkl \
    --output-dir /content/drive/MyDrive/cls_logs \
    --checkpoint_dir /content/drive/MyDrive/cls_ckpts \
    --resume

In [ ]:
# Pivot 3: small CNN for both substrates (no ResNet-18)
!python experiments/45_cls_ci_cifar.py \
    --num_tasks 10 \
    --epochs_per_task 50 \
    --batch_size 128 \
    --n_seeds 1 \
    --num_workers 2 \
    --neocortex_arch small_cnn \
    --lambda_replay_inline 1.0 \
    --cons_epochs 2 \
    --config_name cls_50ep_smallcnn \
    --output-dir /content/drive/MyDrive/cls_logs \
    --checkpoint_dir /content/drive/MyDrive/cls_ckpts \
    --resume

## 7. Results visualisation — compare CLS-CI v2 vs DER

Loads the most-recent pilot logs for both methods (auto-detects whether they're on Drive or local) and plots per-task ACC curves side by side. Run after at least one of the pilots above has finished.

**Note:** the cell picks the most-recent JSON for each method by timestamp, so once a 100-epoch probe finishes its log will replace the 30-epoch one in the plot. If you want to overlay 30ep + 100ep simultaneously, you'll need to copy the cell and edit the `config_name` filter.


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import statistics

log_dir = Path("/content/drive/MyDrive/cls_logs")
if not log_dir.exists():
    log_dir = Path("results/logs/split_cifar100_ci")
if not log_dir.exists():
    print("No results yet — run a pilot first.")
else:
    # Find the most-recent CLS + DER logs separately.
    cls_paths = sorted(log_dir.glob("*45_cifar_cls_ci*.json"))
    der_paths = sorted(log_dir.glob("*46_der_baseline_cifar*.json"))
    cls_paths = [p for p in cls_paths if "smoke" not in p.name]
    der_paths = [p for p in der_paths if "smoke" not in p.name]

    plt.figure(figsize=(9, 5))
    handled_any = False

    if cls_paths:
        with cls_paths[-1].open() as f:
            d_cls = json.load(f)
        seeds = d_cls["per_seed"]
        n_tasks = len(seeds[0]["per_task"])
        cls_means = [
            statistics.fmean(
                s["per_task"][t]["neo_eval_acc"] for s in seeds
            )
            for t in range(n_tasks)
        ]
        plt.plot(range(n_tasks), cls_means, marker="o", label=f"CLS-CI v2 (n={len(seeds)})")
        print(f"CLS-CI v2 final NEO ACC: {statistics.fmean(s['final_neo_acc'] for s in seeds):.3f}")
        handled_any = True

    if der_paths:
        with der_paths[-1].open() as f:
            d_der = json.load(f)
        seeds = d_der["per_seed"]
        n_tasks = len(seeds[0]["per_task"])
        der_means = [
            statistics.fmean(
                s["per_task"][t]["eval_acc"] for s in seeds
            )
            for t in range(n_tasks)
        ]
        plt.plot(range(n_tasks), der_means, marker="s", label=f"DER pure (n={len(seeds)})")
        print(f"DER baseline final ACC: {statistics.fmean(s['final_acc'] for s in seeds):.3f}")
        handled_any = True

    if handled_any:
        plt.xlabel("Task index (after training)")
        plt.ylabel("Class-incremental ACC on classes seen so far")
        plt.title("Split-CIFAR-100 CI — per-task ACC, CLS-CI v2 vs pure DER")
        plt.legend()
        plt.grid(alpha=0.3)
        plt.show()
    else:
        print("No matching logs found in", log_dir)

## 8. Saving results back across Colab sessions

Colab's local filesystem is wiped when the runtime disconnects.  Three options to preserve `results/logs/split_cifar100_ci/` and `results/checkpoints/cifar100_ci/`:

**Option A (recommended): mount Google Drive at session start, redirect outputs there.**

```python
from google.colab import drive
drive.mount('/content/drive')
# Then pass --output-dir /content/drive/MyDrive/cls_logs and
#         --checkpoint_dir /content/drive/MyDrive/cls_ckpts
# to the training command above.
```

**Option B: download manually via the Files panel** when each batch of runs completes.

**Option C: commit + push from Colab** (requires a GitHub Personal Access Token configured in the runtime):

```bash
!git config user.email "you@example.com"
!git config user.name "Your Name"
!git add results/logs/split_cifar100_ci/
!git commit -m "results: Colab pilot run"
# !git push https://<PAT>@github.com/frpatry/continual-synapse-layer.git main
```
